# 2.5 章节实践

## 概述

前四节已经完成了一维 Add。这一节用三个练习检查自己能否独立写出简单的 TileLang Kernel：先把加法换成乘法，再加入一个标量参数，最后把一维数据扩展成二维矩阵。三个练习继续使用 CopyIn → Compute → CopyOut，后一个练习只比前一个多解决一个问题。

## 学习目标

- 复用 Add 的结构写出逐元素 Mul。
- 将一个标量参数搬入 UB，并用于向量计算。
- 根据逻辑块编号处理二维矩阵中的一个 tile。
- 使用完整输出和 PyTorch 参考结果判断实现是否正确。

三个练习的递进关系如下：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">练习</th>
<th style="text-align: left; vertical-align: top;">复用的内容</th>
<th style="text-align: left; vertical-align: top;">本档新增内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">Mul</td>
<td style="text-align: left; vertical-align: top;">一维分块、GM 与 UB 搬运、向量读写</td>
<td style="text-align: left; vertical-align: top;">将 <code>vadd</code> 换成 <code>vmul</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">SAXPY</td>
<td style="text-align: left; vertical-align: top;">Mul 的一维处理过程</td>
<td style="text-align: left; vertical-align: top;">标量参数、广播以及乘加组合</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">2-D Add</td>
<td style="text-align: left; vertical-align: top;">Add 的向量加法</td>
<td style="text-align: left; vertical-align: top;">二维切片和逻辑块映射</td>
</tr>
</tbody>
</table>

下面三个可编辑脚本都包含 Kernel、Host 输入、参数检查和结果校验。为了让学生把注意力放在本档新增内容上，每个脚本只留下少量 TODO，其余代码均已给出并配有中文注释。

**TODO 是需要你补全的代码。** 尚未补全时，执行单元只会提示“尚未执行”，不表示验证通过。

<p><small>每档练习先运行写入单元保存补全后的脚本，再运行执行单元，完成编译、调用与校验。</small></p>

## 1. Mul：只替换计算指令

Mul 的公式是：

```text
out[i] = x[i] * y[i]
```

它的数据形状、分块方式和搬运过程与 2.4 的 Add 相同。唯一需要改变的是 Compute 中的运算：Add 使用 `T.simd.vadd`，Mul 使用 `T.simd.vmul`。

阅读下面的代码，先确认 CopyIn 和 CopyOut 与 Add 相同，再补全 Compute 中的 TODO：

<small>线上没有950设备时，可直接阅读本页结果；自行运行请按1.4准备环境，并从下方准备单元开始顺序执行。辅助代码仅负责启动程序与保存日志。</small>

In [1]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile


def run_example(script):
    """在独立进程中运行，并保留完整日志。"""
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    log_dir = Path(os.environ.get("COURSE_OUTPUT_DIR", tempfile.gettempdir())) / "notebook_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_dir.joinpath(Path(script).stem + ".log").write_text(
        result.stdout + result.stderr, encoding="utf-8"
    )
    # 页内显示程序结果；安装路径、编译过程和底层警告保留在原始日志中。
    print("\n".join(
        line for line in result.stdout.splitlines()
        if not line.startswith("Loading tilelang libs from dev root:")
        and not re.match(r"^\d{4}-\d{2}-\d{2}.*\[TileLang:", line)
    ))
    if result.returncode:
        print(result.stderr)
        result.check_returncode()


In [2]:
%%writefile /tmp/tilelang_ch02_mul_practice.py
import torch
import tilelang
import tilelang.language as T


def make_mul(n: int = 1024, num_blocks: int = 4):
    # 将 n 个元素平均分给 num_blocks 个逻辑块。
    if n <= 0 or num_blocks <= 0 or n % num_blocks:
        raise ValueError("n 和 num_blocks 必须为正数，且 n 能被 num_blocks 整除")
    tile = n // num_blocks
    if tile % 64:
        raise ValueError("每个 tile 的元素数必须是 64 的倍数")

    @T.prim_func
    def main(
        # x、y 是输入，out 是由 Kernel 写入的输出。
        x: T.Tensor((n,), T.float32),
        y: T.Tensor((n,), T.float32),
        out: T.Tensor((n,), T.float32),
    ):
        # 启动 num_blocks 份逻辑任务，bx 是当前逻辑块编号。
        with T.Kernel(num_blocks) as bx:
            # 当前块处理区间 [begin, begin + tile)。
            begin = bx * tile

            # 为当前块在 UB 中分配输入和输出工作区。
            x_ub = T.alloc_shared((tile,), T.float32)
            y_ub = T.alloc_shared((tile,), T.float32)
            out_ub = T.alloc_shared((tile,), T.float32)

            # CopyIn：把当前块负责的 x、y 从 GM 搬到 UB。
            T.copy(x[begin : begin + tile], x_ub)
            T.copy(y[begin : begin + tile], y_ub)

            # Compute：每轮处理 64 个 float32 元素。
            with T.SimdVF():
                mask = T.simd.pset(32)
                for i in T.serial(tile // 64):
                    # 从 UB 读取两个寄存器向量。
                    x_vec = T.simd.vld(x_ub[i * 64])
                    y_vec = T.simd.vld(y_ub[i * 64])

                    # TODO：使用向量乘法计算 x_vec * y_vec。
                    out_vec = ...

                    # 把本轮结果从寄存器写回 UB。
                    T.simd.vsts(out_ub[i * 64], out_vec, mask)

            # CopyOut：将当前 tile 写回 GM 中的对应区间。
            T.copy(out_ub, out[begin : begin + tile])

    return main


def run():
    # Host：构造输入、编译 Kernel，并比较完整输出。
    torch.manual_seed(41)
    x = torch.randn(1024, device="npu", dtype=torch.float32)
    y = torch.randn(1024, device="npu", dtype=torch.float32)

    kernel = tilelang.compile(make_mul(), target="ascend", out_idx=-1)
    actual = kernel(x, y)
    torch.npu.synchronize()

    expected = x * y
    torch.testing.assert_close(actual, expected, rtol=0.0, atol=0.0)
    print("Mul 练习通过")


if __name__ == "__main__":
    run()


Overwriting /tmp/tilelang_ch02_mul_practice.py


In [3]:
from pathlib import Path
import subprocess
import sys

practice = Path("/tmp/tilelang_ch02_mul_practice.py")
if not practice.exists():
    print("请先运行上方代码单元，生成 Mul 练习脚本。")
elif "out_vec = ..." in practice.read_text(encoding="utf-8"):
    print("Mul 尚未执行：请先补全 out_vec 的 TODO，再重新运行两个单元。")
else:
    # 使用独立子进程编译、运行并校验学生补全的 Mul。
    run_example(str(practice))


Mul 尚未执行：请先补全 out_vec 的 TODO，再重新运行两个单元。


这一档只增加一个知识点：`T.simd.vmul(lhs, rhs, mask)` 对两个寄存器向量逐元素相乘。

## 2. SAXPY：让标量参与向量计算

SAXPY 的名称来自 **Single-precision A times X plus Y**，表示“单精度标量乘向量，再加另一个向量”。它的公式是：

```text
out[i] = alpha * x[i] + y[i]
```

与 Mul 相比，它增加了一个标量 `alpha`。本节用 shape 为 `(1,)` 的 Tensor 传入该标量。它进入向量计算需要经过三步：

1. 使用 `T.copy` 将 `alpha` 从 GM 搬到 `alpha_ub`。
2. 使用 `BRC_B32` 将一个 float32 值广播成寄存器向量。
3. 先用 `vmul` 计算 `alpha*x`，再用 `vadd` 加上 `y`。

In [4]:
%%writefile /tmp/tilelang_ch02_saxpy_practice.py
import torch
import tilelang
import tilelang.language as T


def make_saxpy(n: int = 1024, num_blocks: int = 4):
    # SAXPY 沿用 Mul 的一维均分策略。
    if n <= 0 or num_blocks <= 0 or n % num_blocks:
        raise ValueError("n 和 num_blocks 必须为正数，且 n 能被 num_blocks 整除")
    tile = n // num_blocks
    if tile % 64:
        raise ValueError("每个 tile 的元素数必须是 64 的倍数")

    @T.prim_func
    def main(
        # alpha 使用 shape=(1,) 的 Tensor 表示一个标量输入。
        x: T.Tensor((n,), T.float32),
        y: T.Tensor((n,), T.float32),
        alpha: T.Tensor((1,), T.float32),
        out: T.Tensor((n,), T.float32),
    ):
        with T.Kernel(num_blocks) as bx:
            # 当前块负责一个连续的一维区间。
            begin = bx * tile

            # 为三个输入和输出分别分配 UB 工作区。
            x_ub = T.alloc_shared((tile,), T.float32)
            y_ub = T.alloc_shared((tile,), T.float32)
            alpha_ub = T.alloc_shared((1,), T.float32)
            out_ub = T.alloc_shared((tile,), T.float32)

            # CopyIn：向量切片和标量都先从 GM 搬入 UB。
            T.copy(x[begin : begin + tile], x_ub)
            T.copy(y[begin : begin + tile], y_ub)
            T.copy(alpha[:], alpha_ub)

            with T.SimdVF():
                mask = T.simd.pset(32)

                # BRC_B32 把一个 alpha 广播到 64 个 float32 lane。
                alpha_vec = T.simd.vld(alpha_ub[0], "BRC_B32")

                # 每轮处理 x、y 中连续的 64 个元素。
                for i in T.serial(tile // 64):
                    x_vec = T.simd.vld(x_ub[i * 64])
                    y_vec = T.simd.vld(y_ub[i * 64])

                    # TODO：先计算 alpha*x，再加上 y。
                    scaled_vec = ...
                    out_vec = ...

                    # 将本轮 SAXPY 结果写回输出工作区。
                    T.simd.vsts(out_ub[i * 64], out_vec, mask)

            # CopyOut：将当前块的结果写回全局输出 Tensor。
            T.copy(out_ub, out[begin : begin + tile])

    return main


def run():
    # Host：使用固定输入验证 out = alpha*x+y 的完整结果。
    torch.manual_seed(41)
    x = torch.randn(1024, device="npu", dtype=torch.float32)
    y = torch.randn(1024, device="npu", dtype=torch.float32)
    alpha = torch.tensor([0.25], device="npu", dtype=torch.float32)

    kernel = tilelang.compile(make_saxpy(), target="ascend", out_idx=-1)
    actual = kernel(x, y, alpha)
    torch.npu.synchronize()

    expected = alpha[0] * x + y
    torch.testing.assert_close(actual, expected, rtol=0.0, atol=0.0)
    print("SAXPY 练习通过")


if __name__ == "__main__":
    run()


Overwriting /tmp/tilelang_ch02_saxpy_practice.py


In [5]:
from pathlib import Path
import subprocess
import sys

practice = Path("/tmp/tilelang_ch02_saxpy_practice.py")
source = practice.read_text(encoding="utf-8") if practice.exists() else ""
if not practice.exists():
    print("请先运行上方代码单元，生成 SAXPY 练习脚本。")
elif "scaled_vec = ..." in source or "out_vec = ..." in source:
    print("SAXPY 尚未执行：请先补全两个 TODO，再重新运行两个单元。")
else:
    # 使用独立子进程编译、运行并校验学生补全的 SAXPY。
    run_example(str(practice))


SAXPY 尚未执行：请先补全两个 TODO，再重新运行两个单元。


`BRC_B32` 的作用是把 `alpha_ub[0]` 中的一个 32 位值复制到各个 lane。它没有改变 `alpha` 的数学含义，只是让标量能够参加一组元素的向量计算。

## 3. 2-D Add：从一维区间扩展到二维切片

最后一个练习仍然做加法，但输入变成两个 `(16,128)` 的矩阵：

```text
C[i, j] = A[i, j] + B[i, j]
```

本例把矩阵沿列方向切成两个 `(16,64)` tile。两个逻辑块执行相同代码，`bid` 决定当前处理哪一个 tile。

<img alt="二维 Add 从 bid 定位矩阵切片并逐行计算" src="images/02.05_practice_v2.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

图中先用 `bid` 算出当前切片的行列起点，再展开 `bid=1` 的处理：从 A、B 取得行 `[0,16)`、列 `[64,128)` 的数据，分别搬入 `(16,64)` 的 UB 工作区，逐行完成向量加法，并写回 C 的相同位置。`bid=0` 使用同一份代码，处理列 `[0,64)`；两块合起来覆盖完整输出。

为了让写法也适用于更多二维切分，可以先计算每个方向包含多少个 tile，再把一维的 `bid` 转换成行列起点：

```python
row_tiles = m // tile_m
col_tiles = n // tile_n

row0 = (bid // col_tiles) * tile_m
col0 = (bid % col_tiles) * tile_n
```

完整 Kernel 结构如下。接口、UB 和搬运都已经给出，只需补全逐行向量加法：

In [6]:
%%writefile /tmp/tilelang_ch02_2d_add_practice.py
import torch
import tilelang
import tilelang.language as T


def make_2d_add(
    m: int = 16,
    n: int = 128,
    tile_m: int = 16,
    tile_n: int = 64,
):
    # 当前练习只处理能够被 tile 完整切分的矩阵。
    if min(m, n, tile_m, tile_n) <= 0 or m % tile_m or n % tile_n:
        raise ValueError("矩阵 shape 必须能被 tile 完整切分")
    if tile_n % 64:
        raise ValueError("tile_n 必须是 64 的倍数")

    # 计算矩阵沿行、列方向各包含多少个 tile。
    row_tiles = m // tile_m
    col_tiles = n // tile_n

    @T.prim_func
    def main(
        a: T.Tensor((m, n), T.float32),
        b: T.Tensor((m, n), T.float32),
        c: T.Tensor((m, n), T.float32),
    ):
        # 每个逻辑块负责一个二维 tile。
        with T.Kernel(row_tiles * col_tiles) as bid:
            # 将一维 bid 转换成当前 tile 左上角的行、列坐标。
            row0 = (bid // col_tiles) * tile_m
            col0 = (bid % col_tiles) * tile_n

            # UB 工作区的 shape 与当前 tile 相同。
            a_ub = T.alloc_shared((tile_m, tile_n), T.float32)
            b_ub = T.alloc_shared((tile_m, tile_n), T.float32)
            c_ub = T.alloc_shared((tile_m, tile_n), T.float32)

            # CopyIn：从 A、B 中截取当前二维 tile。
            T.copy(a[row0 : row0 + tile_m, col0 : col0 + tile_n], a_ub)
            T.copy(b[row0 : row0 + tile_m, col0 : col0 + tile_n], b_ub)

            with T.SimdVF():
                mask = T.simd.pset(32)

                # 外层循环选行，内层循环每次处理该行中的 64 个元素。
                for i in T.serial(tile_m):
                    for k in T.serial(tile_n // 64):
                        # TODO：读取 A、B 的向量，完成加法并写回 c_ub。
                        a_vec = ...
                        b_vec = ...
                        c_vec = ...
                        ...

            # CopyOut：将 c_ub 写回 C 中当前 tile 对应的位置。
            T.copy(c_ub, c[row0 : row0 + tile_m, col0 : col0 + tile_n])

    return main


def run():
    # Host：构造二维输入并比较矩阵中的每一个输出元素。
    torch.manual_seed(41)
    a = torch.randn(16, 128, device="npu", dtype=torch.float32)
    b = torch.randn(16, 128, device="npu", dtype=torch.float32)

    kernel = tilelang.compile(make_2d_add(), target="ascend", out_idx=-1)
    actual = kernel(a, b)
    torch.npu.synchronize()

    expected = a + b
    torch.testing.assert_close(actual, expected, rtol=0.0, atol=0.0)
    print("2-D Add 练习通过")


if __name__ == "__main__":
    run()


Overwriting /tmp/tilelang_ch02_2d_add_practice.py


In [7]:
from pathlib import Path
import subprocess
import sys

practice = Path("/tmp/tilelang_ch02_2d_add_practice.py")
source = practice.read_text(encoding="utf-8") if practice.exists() else ""
todo_lines = ("a_vec = ...", "b_vec = ...", "c_vec = ...")
if not practice.exists():
    print("请先运行上方代码单元，生成 2-D Add 练习脚本。")
elif any(line in source for line in todo_lines) or "\n                        ...\n" in source:
    print("2-D Add 尚未执行：请先补全四处 TODO，再重新运行两个单元。")
else:
    # 使用独立子进程编译、运行并校验学生补全的 2-D Add。
    run_example(str(practice))


2-D Add 尚未执行：请先补全四处 TODO，再重新运行两个单元。


这里的外层循环依次选择 tile 中的 16 行，内层循环每次处理一组 64 个 float32 元素。当前参数下 `tile_n=64`，所以每一行只执行一次内层循环。

## 4. 用完整输出验收三个练习

Host 侧为三个练习准备固定输入，并将每个完整输出与对应的 PyTorch 表达式比较：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">练习</th>
<th style="text-align: left; vertical-align: top;">输入</th>
<th style="text-align: left; vertical-align: top;">PyTorch 参考结果</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">Mul</td>
<td style="text-align: left; vertical-align: top;"><code>x, y: (1024,) float32</code></td>
<td style="text-align: left; vertical-align: top;"><code>x * y</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">SAXPY</td>
<td style="text-align: left; vertical-align: top;"><code>x, y: (1024,) float32</code>，<code>alpha: (1,) float32</code></td>
<td style="text-align: left; vertical-align: top;"><code>alpha[0] * x + y</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">2-D Add</td>
<td style="text-align: left; vertical-align: top;"><code>a, b: (16,128) float32</code></td>
<td style="text-align: left; vertical-align: top;"><code>a + b</code></td>
</tr>
</tbody>
</table>

三个结果都使用：

```python
torch.testing.assert_close(actual, expected, rtol=0.0, atol=0.0)
```

下面的执行单元会启动 `src/02.05_answer.py`，按照 **Mul → SAXPY → 2-D Add** 的顺序串行执行三个算子。每个算子都会依次完成编译、调用、设备同步和完整输出比较；前一个算子通过后，脚本才会继续下一个。任何一步失败，执行单元都会立即报错。



**下面展示的是课程完整答案的验证结果，不代表上方的 TODO 练习已经完成。**

完成练习后，可以对照[完整代码](answer/02.05_answer.py)和[逐档解析](answer/02.05_questions.txt)检查自己的实现。

In [8]:
import subprocess, sys

# 按 Mul → SAXPY → 2-D Add 的顺序串行运行课程完整答案。
run_example("src/02.05_answer.py")

Mul 校验通过：1024 个元素被分为 4 个 tile
SAXPY 校验通过：标量 alpha 参与全部元素计算
2-D Add 校验通过：(16, 128) 被分为两个 (16, 64) tile
Verification passed!


## 小结

三个练习始终沿用 CopyIn → Compute → CopyOut。Mul 只替换计算指令；SAXPY 增加标量的搬运、广播和乘加；2-D Add 将一维区间扩展成二维 tile。完成三档练习并通过完整输出比较，说明已经能够独立写出基础的 TileLang Vector Kernel。

下一章会继续解决一个问题：当输入规模和片上空间发生变化时，怎样系统地确定 tile 大小、任务数量和尾块处理方式。

## 课后练习

不查看完整答案，依次补全本节三个 TODO。每完成一档，先用对应的 PyTorch 参考结果检查完整输出，再继续下一档。